In [1]:
import os

PROCESSED = "/kaggle/input/datasets/chaitanyanemade/df2-processed/processed"

print("Checking processed files from Notebook 1...\n")
required = [
    "manifest_train.json",
    "manifest_val.json",
    "manifest_test.json",
    "class_weights.json",
    "label_map.json",
]
all_ok = True
for fname in required:
    path = os.path.join(PROCESSED, fname)
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {fname}")
    if not exists:
        all_ok = False

if not all_ok:
    print("\n✗ Some files missing!")
else:
    print("\n✓ All required files found. Ready to train.")

Checking processed files from Notebook 1...

  ✓ manifest_train.json
  ✓ manifest_val.json
  ✓ manifest_test.json
  ✓ class_weights.json
  ✓ label_map.json

✓ All required files found. Ready to train.


In [2]:
# ============================================================
# CELL 1 — Install / imports
# ============================================================

!pip install timm --quiet
!pip install scikit-learn --quiet

import os, json, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    classification_report
)
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

# ── Paths ─────────────────────────────────────────────────────
PROCESSED  = "/kaggle/input/datasets/chaitanyanemade/df2-processed/processed"
OUTPUT_DIR = "/kaggle/working/clf_results"
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")

os.makedirs(CKPT_DIR, exist_ok=True)

print("\n✅ Imports done. Ready for config.")

Device: cuda
GPU   : Tesla T4

✅ Imports done. Ready for config.


In [3]:
# ============================================================
# CELL 2 — Config (edit here if needed)
# ============================================================

# ── Classes ───────────────────────────────────────────────────
LABEL_MAP = {
    "short sleeve top": 0,
    "trousers":         1,
    "shorts":           2,
    "long sleeve top":  3,
    "skirt":            4,
}
CLASS_NAMES = [k for k, v in sorted(LABEL_MAP.items(), key=lambda x: x[1])]
NUM_CLASSES = len(CLASS_NAMES)

# ── Training hyperparameters ──────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32      # If OOM later → reduce to 16
NUM_EPOCHS  = 30

LR_SCRATCH  = 1e-3
LR_TL       = 1e-3
LR_FT       = 1e-4

PATIENCE    = 5

# ── Load class weights ────────────────────────────────────────
with open(os.path.join(PROCESSED, "class_weights.json")) as f:
    cw = json.load(f)

POS_WEIGHT = torch.tensor(cw["pos_weight"], dtype=torch.float32).to(DEVICE)

print("✅ Config set.\n")
print(f"Classes     : {CLASS_NAMES}")
print(f"Num classes : {NUM_CLASSES}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"Pos weights : {[round(w, 2) for w in cw['pos_weight']]}")

✅ Config set.

Classes     : ['short sleeve top', 'trousers', 'shorts', 'long sleeve top', 'skirt']
Num classes : 5
Batch size  : 32
Epochs      : 30
Pos weights : [1.07, 1.61, 3.01, 2.95, 3.7]


In [4]:
# ============================================================
# CELL 3 — Dataset class + DataLoaders
# ============================================================

class FashionDataset(Dataset):
    def __init__(self, manifest_path, transform=None):
        with open(manifest_path) as f:
            self.records = json.load(f)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]

        try:
            img = Image.open(rec["image_path"]).convert("RGB")
        except Exception:
            # fallback if image missing/corrupt
            img = Image.new("RGB", (224, 224), (128, 128, 128))

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(rec["multilabel"], dtype=torch.float32)
        return img, label


# ── Transforms ────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


# ── Load datasets ─────────────────────────────────────────────
train_ds = FashionDataset(os.path.join(PROCESSED, "manifest_train.json"), train_transform)
val_ds   = FashionDataset(os.path.join(PROCESSED, "manifest_val.json"),   val_transform)
test_ds  = FashionDataset(os.path.join(PROCESSED, "manifest_test.json"),  val_transform)

# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)


# ── Quick sanity check ────────────────────────────────────────
print(f"Train size : {len(train_ds)}")
print(f"Val size   : {len(val_ds)}")
print(f"Test size  : {len(test_ds)}")
print(f"Train batches: {len(train_loader)}")

# Check one batch
imgs, labels = next(iter(train_loader))
print(f"\nBatch shape (images): {imgs.shape}")
print(f"Batch shape (labels): {labels.shape}")

Train size : 9601
Val size   : 1199
Test size  : 1200
Train batches: 301

Batch shape (images): torch.Size([32, 3, 224, 224])
Batch shape (labels): torch.Size([32, 5])


In [5]:
# ============================================================
# CELL 4 — Model builder
# ============================================================

def build_model(model_name, strategy, num_classes=NUM_CLASSES):
    """
    model_name : "resnet50" | "efficientnet_b0" | "mobilenetv3"
    strategy   : "scratch" | "tl" | "ft"
    """

    pretrained = (strategy != "scratch")

    # ── ResNet50 ──────────────────────────────────────────────
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        model   = models.resnet50(weights=weights)

        in_feat = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_feat, num_classes)
        )

        backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
        head_params     = list(model.fc.parameters())

    # ── EfficientNet-B0 ───────────────────────────────────────
    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model   = models.efficientnet_b0(weights=weights)

        in_feat = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_feat, num_classes)
        )

        backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
        head_params     = list(model.classifier.parameters())

    # ── MobileNetV3 ───────────────────────────────────────────
    elif model_name == "mobilenetv3":
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
        model   = models.mobilenet_v3_large(weights=weights)

        in_feat = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_feat, num_classes)

        backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
        head_params     = list(model.classifier.parameters())

    else:
        raise ValueError("Invalid model name")

    # ── Freeze backbone for TL & FT initially ─────────────────
    if strategy in ["tl", "ft"]:
        for p in backbone_params:
            p.requires_grad = False

    model = model.to(DEVICE)

    return model, backbone_params, head_params


def unfreeze_backbone(model, backbone_params):
    for p in backbone_params:
        p.requires_grad = True
    print("✅ Backbone unfrozen")


print("✅ Model builder ready.")

✅ Model builder ready.


In [6]:
# ============================================================
# CELL 5 — Training and evaluation functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0.0

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # sigmoid
    all_probs = 1 / (1 + np.exp(-all_logits))
    all_preds = (all_probs >= threshold).astype(int)

    avg_loss = total_loss / len(loader.dataset)

    # ── Metrics ───────────────────────────────────────────────
    metrics = {}

    for i, cls in enumerate(CLASS_NAMES):
        metrics[cls] = {
            "precision": precision_score(all_labels[:, i], all_preds[:, i], zero_division=0),
            "recall":    recall_score(all_labels[:, i], all_preds[:, i], zero_division=0),
            "f1":        f1_score(all_labels[:, i], all_preds[:, i], zero_division=0),
            "auc":       roc_auc_score(all_labels[:, i], all_probs[:, i])
                         if all_labels[:, i].sum() > 0 else 0.0,
        }

    metrics["macro_f1"] = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    metrics["micro_f1"] = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    metrics["macro_auc"] = np.mean([metrics[c]["auc"] for c in CLASS_NAMES])

    return avg_loss, metrics, all_probs, all_labels


def plot_roc_curves(all_probs, all_labels, title, save_path):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 6))

    for i, cls in enumerate(CLASS_NAMES):
        if all_labels[:, i].sum() == 0:
            continue

        fpr, tpr, _ = roc_curve(all_labels[:, i], all_probs[:, i])
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])

        ax.plot(fpr, tpr, lw=2, label=f"{cls} (AUC={auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()

    print(f"ROC saved → {save_path}")


print("✅ Training functions ready.")

✅ Training functions ready.


In [7]:
# ============================================================
# CELL 6 — Main training loop
# ============================================================

def train_model(model_name, strategy):

    run_name = f"{model_name}_{strategy}"
    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name.upper()} | {strategy.upper()}")
    print(f"{'='*60}")

    # ── Build model ───────────────────────────────────────────
    model, backbone_params, head_params = build_model(model_name, strategy)

    # ── Loss ──────────────────────────────────────────────────
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

    # ── Optimizer ─────────────────────────────────────────────
    if strategy == "scratch":
        optimizer = optim.Adam(model.parameters(), lr=LR_SCRATCH, weight_decay=1e-4)
    else:
        optimizer = optim.Adam(
            [{"params": head_params, "lr": LR_TL if strategy == "tl" else LR_FT}],
            weight_decay=1e-4
        )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    FT_UNFREEZE_EPOCH = 5

    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):

        # ── Fine-tuning unfreeze ──────────────────────────────
        if strategy == "ft" and epoch == FT_UNFREEZE_EPOCH:
            unfreeze_backbone(model, backbone_params)
            optimizer.add_param_group({
                "params": backbone_params,
                "lr": LR_FT * 0.1
            })

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader, criterion)

        scheduler.step()

        val_f1 = val_metrics["macro_f1"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        elapsed = (time.time() - start_time) / 60

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val F1: {val_f1:.4f} | "
              f"Time: {elapsed:.1f}m")

        # ── Save best model ───────────────────────────────────
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

            torch.save(model.state_dict(),
                       os.path.join(CKPT_DIR, f"{run_name}_best.pth"))

        else:
            patience_counter += 1

            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    total_time = (time.time() - start_time) / 60

    print(f"\nTraining complete in {total_time:.1f} min")
    print(f"Best Val F1: {best_val_f1:.4f}")

    # ── Load best model ───────────────────────────────────────
    model.load_state_dict(best_weights)

    # ── Test evaluation ───────────────────────────────────────
    test_loss, test_metrics, test_probs, test_labels = evaluate(
        model, test_loader, criterion
    )

    print("\nTEST RESULTS:")
    print(f"Macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"Micro-F1: {test_metrics['micro_f1']:.4f}")
    print(f"Macro-AUC: {test_metrics['macro_auc']:.4f}")

    return {
        "model": model_name,
        "strategy": strategy,
        "macro_f1": test_metrics["macro_f1"],
        "micro_f1": test_metrics["micro_f1"],
        "macro_auc": test_metrics["macro_auc"],
        "time": total_time
    }


print("✅ Training loop ready.")

✅ Training loop ready.


In [8]:
# ============================================================
# CELL 7 — TEST RUN (EfficientNet TL only)
# ============================================================

result = train_model("efficientnet_b0", "tl")

print("\nTest run complete!")
print(result)


TRAINING: EFFICIENTNET_B0 | TL
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s] 


Epoch 01/30 | Train Loss: 0.8498 | Val Loss: 0.7891 | Val F1: 0.5833 | Time: 1.7m
Epoch 02/30 | Train Loss: 0.7873 | Val Loss: 0.7668 | Val F1: 0.5892 | Time: 2.7m
Epoch 03/30 | Train Loss: 0.7713 | Val Loss: 0.7485 | Val F1: 0.6022 | Time: 3.7m
Epoch 04/30 | Train Loss: 0.7636 | Val Loss: 0.7568 | Val F1: 0.6067 | Time: 4.7m
Epoch 05/30 | Train Loss: 0.7609 | Val Loss: 0.7459 | Val F1: 0.6045 | Time: 5.6m
Epoch 06/30 | Train Loss: 0.7576 | Val Loss: 0.7419 | Val F1: 0.6167 | Time: 6.6m
Epoch 07/30 | Train Loss: 0.7535 | Val Loss: 0.7343 | Val F1: 0.6170 | Time: 7.6m
Epoch 08/30 | Train Loss: 0.7494 | Val Loss: 0.7401 | Val F1: 0.6165 | Time: 8.6m
Epoch 09/30 | Train Loss: 0.7482 | Val Loss: 0.7374 | Val F1: 0.6174 | Time: 9.6m
Epoch 10/30 | Train Loss: 0.7446 | Val Loss: 0.7389 | Val F1: 0.6096 | Time: 10.6m
Epoch 11/30 | Train Loss: 0.7482 | Val Loss: 0.7364 | Val F1: 0.6155 | Time: 11.6m
Epoch 12/30 | Train Loss: 0.7484 | Val Loss: 0.7361 | Val F1: 0.6210 | Time: 12.6m
Epoch 13/30 |

In [9]:
# ============================================================
# FINAL RUN — ALL 9 TRAINING EXPERIMENTS
# ============================================================

MODELS = ["resnet50", "efficientnet_b0", "mobilenetv3"]
STRATEGIES = ["scratch", "tl", "ft"]

all_results = []

for model_name in MODELS:
    for strategy in STRATEGIES:
        result = train_model(model_name, strategy)
        all_results.append(result)
        torch.cuda.empty_cache()

print("\nALL TRAINING RUNS COMPLETE!")


TRAINING: RESNET50 | SCRATCH
Epoch 01/30 | Train Loss: 0.9823 | Val Loss: 0.9563 | Val F1: 0.4088 | Time: 2.0m
Epoch 02/30 | Train Loss: 0.9558 | Val Loss: 1.0730 | Val F1: 0.3815 | Time: 4.0m
Epoch 03/30 | Train Loss: 0.9530 | Val Loss: 0.9642 | Val F1: 0.3399 | Time: 6.0m
Epoch 04/30 | Train Loss: 0.9476 | Val Loss: 0.9563 | Val F1: 0.3525 | Time: 8.0m
Epoch 05/30 | Train Loss: 0.9415 | Val Loss: 1.0244 | Val F1: 0.2557 | Time: 10.0m
Epoch 06/30 | Train Loss: 0.9288 | Val Loss: 0.9813 | Val F1: 0.4097 | Time: 12.0m
Epoch 07/30 | Train Loss: 0.9210 | Val Loss: 0.9560 | Val F1: 0.4604 | Time: 13.9m
Epoch 08/30 | Train Loss: 0.9158 | Val Loss: 0.9660 | Val F1: 0.4723 | Time: 15.9m
Epoch 09/30 | Train Loss: 0.9022 | Val Loss: 0.9552 | Val F1: 0.3944 | Time: 17.9m
Epoch 10/30 | Train Loss: 0.8951 | Val Loss: 0.8983 | Val F1: 0.4857 | Time: 19.8m
Epoch 11/30 | Train Loss: 0.8838 | Val Loss: 0.9024 | Val F1: 0.4901 | Time: 21.8m
Epoch 12/30 | Train Loss: 0.8692 | Val Loss: 0.8911 | Val F1:

100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]


Epoch 01/30 | Train Loss: 0.8362 | Val Loss: 0.7525 | Val F1: 0.6121 | Time: 1.0m
Epoch 02/30 | Train Loss: 0.7750 | Val Loss: 0.7493 | Val F1: 0.6166 | Time: 2.1m
Epoch 03/30 | Train Loss: 0.7618 | Val Loss: 0.7899 | Val F1: 0.5819 | Time: 3.1m
Epoch 04/30 | Train Loss: 0.7600 | Val Loss: 0.7254 | Val F1: 0.6164 | Time: 4.2m
Epoch 05/30 | Train Loss: 0.7565 | Val Loss: 0.7640 | Val F1: 0.5702 | Time: 5.2m
Epoch 06/30 | Train Loss: 0.7520 | Val Loss: 0.7816 | Val F1: 0.6099 | Time: 6.3m
Epoch 07/30 | Train Loss: 0.7557 | Val Loss: 0.7296 | Val F1: 0.5981 | Time: 7.3m
Early stopping at epoch 7

Training complete in 7.3 min
Best Val F1: 0.6166

TEST RESULTS:
Macro-F1: 0.6038
Micro-F1: 0.6182
Macro-AUC: 0.8050

TRAINING: RESNET50 | FT
Epoch 01/30 | Train Loss: 0.9096 | Val Loss: 0.8657 | Val F1: 0.5669 | Time: 1.0m
Epoch 02/30 | Train Loss: 0.8466 | Val Loss: 0.8229 | Val F1: 0.5704 | Time: 2.1m
Epoch 03/30 | Train Loss: 0.8149 | Val Loss: 0.7993 | Val F1: 0.6084 | Time: 3.1m
Epoch 04/30 

100%|██████████| 21.1M/21.1M [00:00<00:00, 62.6MB/s]


Epoch 01/30 | Train Loss: 0.8007 | Val Loss: 0.7870 | Val F1: 0.5613 | Time: 1.0m
Epoch 02/30 | Train Loss: 0.7289 | Val Loss: 0.7153 | Val F1: 0.6295 | Time: 2.1m
Epoch 03/30 | Train Loss: 0.6890 | Val Loss: 0.7083 | Val F1: 0.6381 | Time: 3.1m
Epoch 04/30 | Train Loss: 0.6617 | Val Loss: 0.7182 | Val F1: 0.6088 | Time: 4.2m
Epoch 05/30 | Train Loss: 0.6332 | Val Loss: 0.7585 | Val F1: 0.5820 | Time: 5.3m
Epoch 06/30 | Train Loss: 0.6148 | Val Loss: 0.7258 | Val F1: 0.6141 | Time: 6.3m
Epoch 07/30 | Train Loss: 0.5901 | Val Loss: 0.7508 | Val F1: 0.6283 | Time: 7.4m
Epoch 08/30 | Train Loss: 0.5719 | Val Loss: 0.6954 | Val F1: 0.6432 | Time: 8.5m
Epoch 09/30 | Train Loss: 0.5410 | Val Loss: 0.7128 | Val F1: 0.6416 | Time: 9.5m
Epoch 10/30 | Train Loss: 0.5139 | Val Loss: 0.7271 | Val F1: 0.6380 | Time: 10.6m
Epoch 11/30 | Train Loss: 0.4904 | Val Loss: 0.7648 | Val F1: 0.6318 | Time: 11.6m
Epoch 12/30 | Train Loss: 0.4689 | Val Loss: 0.7863 | Val F1: 0.6271 | Time: 12.7m
Epoch 13/30 |

In [10]:
# Recreate all_results manually from your outputs
# Safe re-imports
import pandas as pd
import os
all_results = [
    {"model":"resnet50","strategy":"scratch","macro_f1":0.5742,"micro_f1":0.5771,"macro_auc":0.7503,"time":49.2},
    {"model":"resnet50","strategy":"tl","macro_f1":0.6038,"micro_f1":0.6182,"macro_auc":0.8050,"time":7.4},
    {"model":"resnet50","strategy":"ft","macro_f1":0.7792,"micro_f1":0.7891,"macro_auc":0.9183,"time":25.7},

    {"model":"efficientnet_b0","strategy":"scratch","macro_f1":0.4471,"micro_f1":0.4582,"macro_auc":0.5194,"time":8.9},
    {"model":"efficientnet_b0","strategy":"tl","macro_f1":0.6228,"micro_f1":0.6356,"macro_auc":0.8088,"time":21.5},
    {"model":"efficientnet_b0","strategy":"ft","macro_f1":0.7439,"micro_f1":0.7500,"macro_auc":0.8893,"time":31.4},

    {"model":"mobilenetv3","strategy":"scratch","macro_f1":0.5968,"micro_f1":0.5993,"macro_auc":0.7644,"time":30.9},
    {"model":"mobilenetv3","strategy":"tl","macro_f1":0.6584,"micro_f1":0.6724,"macro_auc":0.8367,"time":14.4},
    {"model":"mobilenetv3","strategy":"ft","macro_f1":0.7127,"micro_f1":0.7228,"macro_auc":0.8834,"time":26.5},
]

print("✅ all_results recreated")

✅ all_results recreated


In [11]:
# ============================================================
# CELL 8 — Build comparison table
# ============================================================

rows = []

for r in all_results:
    row = {
        "Model":      r["model"],
        "Strategy":   r["strategy"],
        "Macro-F1":   round(r["macro_f1"],  4),
        "Micro-F1":   round(r["micro_f1"],  4),
        "Macro-AUC":  round(r["macro_auc"], 4),
        "Train_min":  round(r["time"], 1)
    }
    rows.append(row)

df_results = pd.DataFrame(rows)

# Sort by best model
df_results = df_results.sort_values("Macro-F1", ascending=False)

print("\n" + "="*70)
print("FINAL RESULTS — SORTED BY MACRO-F1")
print("="*70)

print(df_results.to_string(index=False))

# Save CSV (for report)
csv_path = os.path.join(OUTPUT_DIR, "classification_results.csv")
df_results.to_csv(csv_path, index=False)

print(f"\n✅ Results saved at: {csv_path}")


FINAL RESULTS — SORTED BY MACRO-F1
          Model Strategy  Macro-F1  Micro-F1  Macro-AUC  Train_min
       resnet50       ft    0.7792    0.7891     0.9183       25.7
efficientnet_b0       ft    0.7439    0.7500     0.8893       31.4
    mobilenetv3       ft    0.7127    0.7228     0.8834       26.5
    mobilenetv3       tl    0.6584    0.6724     0.8367       14.4
efficientnet_b0       tl    0.6228    0.6356     0.8088       21.5
       resnet50       tl    0.6038    0.6182     0.8050        7.4
    mobilenetv3  scratch    0.5968    0.5993     0.7644       30.9
       resnet50  scratch    0.5742    0.5771     0.7503       49.2
efficientnet_b0  scratch    0.4471    0.4582     0.5194        8.9

✅ Results saved at: /kaggle/working/clf_results/classification_results.csv


In [12]:
# ============================================================
# CELL 9 — Comparison bar chart
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Extract values
models     = df_results["Model"].tolist()
strategies = df_results["Strategy"].tolist()
macro_f1s  = df_results["Macro-F1"].tolist()

x = np.arange(len(models))

# Colors for strategies
color_map = {
    "scratch": "#B4B2A9",
    "tl": "#3B8BD4",
    "ft": "#1D9E75"
}
colors = [color_map[s] for s in strategies]

# Plot
plt.figure(figsize=(12, 5))
plt.bar(x, macro_f1s, color=colors)

# Add values on top
for i, v in enumerate(macro_f1s):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)

# Labels
plt.xticks(x, [f"{m}\n({s})" for m, s in zip(models, strategies)], fontsize=8)
plt.ylabel("Macro-F1")
plt.title("Model Comparison (Macro-F1)")

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=c, label=s) for s, c in color_map.items()]
plt.legend(handles=legend, title="Strategy")

plt.ylim(0, 1.0)
plt.tight_layout()

# Save
chart_path = os.path.join(OUTPUT_DIR, "model_comparison_bar.png")
plt.savefig(chart_path, dpi=100)

plt.show()

print(f"✅ Chart saved at: {chart_path}")

✅ Chart saved at: /kaggle/working/clf_results/model_comparison_bar.png


In [13]:
from torchvision import models
train_model("resnet50", "ft")


TRAINING: RESNET50 | FT
Epoch 01/30 | Train Loss: 0.9215 | Val Loss: 0.8832 | Val F1: 0.5167 | Time: 1.2m
Epoch 02/30 | Train Loss: 0.8533 | Val Loss: 0.8300 | Val F1: 0.5758 | Time: 2.3m
Epoch 03/30 | Train Loss: 0.8208 | Val Loss: 0.8033 | Val F1: 0.5824 | Time: 3.5m
Epoch 04/30 | Train Loss: 0.7983 | Val Loss: 0.7924 | Val F1: 0.5847 | Time: 4.6m
✅ Backbone unfrozen
Epoch 05/30 | Train Loss: 0.6916 | Val Loss: 0.5889 | Val F1: 0.7030 | Time: 6.6m
Epoch 06/30 | Train Loss: 0.5688 | Val Loss: 0.5294 | Val F1: 0.7383 | Time: 8.6m
Epoch 07/30 | Train Loss: 0.4982 | Val Loss: 0.4946 | Val F1: 0.7536 | Time: 10.6m
Epoch 08/30 | Train Loss: 0.4412 | Val Loss: 0.4712 | Val F1: 0.7809 | Time: 12.6m
Epoch 09/30 | Train Loss: 0.3991 | Val Loss: 0.4643 | Val F1: 0.7851 | Time: 14.6m
Epoch 10/30 | Train Loss: 0.3547 | Val Loss: 0.4647 | Val F1: 0.7945 | Time: 16.6m
Epoch 11/30 | Train Loss: 0.3245 | Val Loss: 0.4721 | Val F1: 0.7867 | Time: 18.6m
Epoch 12/30 | Train Loss: 0.2936 | Val Loss: 0.4

{'model': 'resnet50',
 'strategy': 'ft',
 'macro_f1': 0.774137143202275,
 'micro_f1': 0.7831202046035806,
 'macro_auc': np.float64(0.9127270370272502),
 'time': 44.53638649781545}

In [14]:
# ============================================================
# CELL 10 — Find and save the best model
# ============================================================

# Find best model based on Macro-F1
best_run = max(all_results, key=lambda r: r["macro_f1"])

print("🏆 BEST MODEL FOUND:\n")
print(f"Model    : {best_run['model']}")
print(f"Strategy : {best_run['strategy']}")
print(f"Macro-F1 : {best_run['macro_f1']:.4f}")
print(f"Micro-F1 : {best_run['micro_f1']:.4f}")
print(f"Macro-AUC: {best_run['macro_auc']:.4f}")

# Paths
best_ckpt_src = os.path.join(
    CKPT_DIR,
    f"{best_run['model']}_{best_run['strategy']}_best.pth"
)

best_ckpt_dst = os.path.join(
    OUTPUT_DIR,
    "best_classification_model.pth"
)

# Copy best model
import shutil
shutil.copy2(best_ckpt_src, best_ckpt_dst)

print(f"\n✅ Best model saved at:")
print(best_ckpt_dst)

# Save metadata (for report + HuggingFace)
best_info = {
    "task":        "classification",
    "model":       best_run["model"],
    "strategy":    best_run["strategy"],
    "macro_f1":    best_run["macro_f1"],
    "micro_f1":    best_run["micro_f1"],
    "macro_auc":   best_run["macro_auc"],
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "img_size":    IMG_SIZE,
    "checkpoint":  "best_classification_model.pth"
}

info_path = os.path.join(OUTPUT_DIR, "best_model_info.json")

with open(info_path, "w") as f:
    json.dump(best_info, f, indent=4)

print(f"\n✅ Metadata saved at:")
print(info_path)

🏆 BEST MODEL FOUND:

Model    : resnet50
Strategy : ft
Macro-F1 : 0.7792
Micro-F1 : 0.7891
Macro-AUC: 0.9183

✅ Best model saved at:
/kaggle/working/clf_results/best_classification_model.pth

✅ Metadata saved at:
/kaggle/working/clf_results/best_model_info.json


In [15]:
inference_code = '''
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import json, os
 
CLASS_NAMES = ["short sleeve top", "trousers", "shorts", "long sleeve top", "skirt"]
IMG_SIZE    = 224
THRESHOLD   = 0.5
 
def load_model(model_name, checkpoint_path, num_classes=5):
    """Load model from checkpoint."""
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        in_f  = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    elif model_name == "mobilenetv3":
        model = models.mobilenet_v3_large(weights=None)
        in_f  = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_f, num_classes)
 
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    model.eval()
    return model
 
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
 
def predict(image_path, model):
    """
    Predict clothing classes present in an image.
    Returns: dict with class names and binary predictions.
    """
    img    = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.sigmoid(logits).squeeze().numpy()
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "predictions": preds.tolist(),    # [0/1 per class]
        "probabilities": probs.tolist(),  # raw probabilities
        "classes": CLASS_NAMES,
        "label_map": {name: i for i, name in enumerate(CLASS_NAMES)},
    }
'''
 
inf_path = os.path.join(OUTPUT_DIR, "inference_classification.py")
with open(inf_path, "w") as f:
    f.write(inference_code)
 
print(f"Inference script saved: {inf_path}")

Inference script saved: /kaggle/working/clf_results/inference_classification.py


In [16]:
print("=" * 65)
print("NOTEBOOK 2 COMPLETE — Classification Training & Evaluation")
print("=" * 65)
 
output_files = {
    "checkpoints/":                    "9 model checkpoints (one per run)",
    "best_classification_model.pth":   "Best model → upload to HuggingFace",
    "best_model_info.json":            "Best model metadata",
    "classification_results.csv":      "Full results table — use in report",
    "clf_comparison_bar.png":          "Bar chart for report",
    "inference_classification.py":     "Inference script for HuggingFace",
    "*_roc.png (9 files)":             "ROC curves per model/strategy",
    "*_history.png (9 files)":         "Training curves per model/strategy",
    "*_metrics.json (9 files)":        "Raw metrics per run",
}
 
for fname, desc in output_files.items():
    path   = os.path.join(OUTPUT_DIR, fname.replace(" (9 files)", "").replace("*_roc", "resnet50_scratch_roc"))
    exists = os.path.exists(path)
    mark   = "✓" if exists else "~"
    print(f"  {mark}  {fname:<45} {desc}")
 
print(f"\nBest model: {best_run['model']} ({best_run['strategy']})")
print(f"  Macro-F1 : {best_run['macro_f1']:.4f}")
print(f"  Micro-F1 : {best_run['micro_f1']:.4f}")
print(f"  Macro-AUC: {best_run['macro_auc']:.4f}")
print("\nNext: Run Notebook 3 — YOLO Detection + Segmentation")

NOTEBOOK 2 COMPLETE — Classification Training & Evaluation
  ✓  checkpoints/                                  9 model checkpoints (one per run)
  ✓  best_classification_model.pth                 Best model → upload to HuggingFace
  ✓  best_model_info.json                          Best model metadata
  ✓  classification_results.csv                    Full results table — use in report
  ~  clf_comparison_bar.png                        Bar chart for report
  ✓  inference_classification.py                   Inference script for HuggingFace
  ~  *_roc.png (9 files)                           ROC curves per model/strategy
  ~  *_history.png (9 files)                       Training curves per model/strategy
  ~  *_metrics.json (9 files)                      Raw metrics per run

Best model: resnet50 (ft)
  Macro-F1 : 0.7792
  Micro-F1 : 0.7891
  Macro-AUC: 0.9183

Next: Run Notebook 3 — YOLO Detection + Segmentation
